# Linear Regression — Training

এই notebook-এ আমরা আগের `Linear Regression: Vector Form` থেকে পরের ধাপে যাব:

> **Prediction করার জন্য `Xw` জানি। কিন্তু `w` কোথা থেকে আসবে?**

এখানেই **training** শুরু।

---

## 1. Model বনাম Training

আগের lesson-এ আমরা শিখেছি:

$$
\hat y = Xw
$$

এখানে:
- $X$ = feature matrix
- $w$ = weights
- $\hat y$ = predictions

কিন্তু training-এর শুরুতে আমাদের কাছে **ভালো weights $w$ জানা থাকে না**।

আমাদের কাছে থাকে:

$$
X \quad\text{এবং}\quad y
$$

যেখানে:
- $y$ = actual target / real price
- $X$ = input features

আমাদের goal:

$$
\boxed{\text{এমন }w\text{ খুঁজে বের করা যাতে }Xw\approx y}
$$

অর্থাৎ predictions যেন actual values-এর যত কাছে সম্ভব হয়।

# 2. একটি ছোট Example

ধরি আমাদের কাছে 5টি car আছে।

প্রতিটি row:

$$
[1,\text{engine\_hp},\text{age},\text{mileage}]
$$

`1` হলো আগের lesson-এর **fictional feature** — এর weight হলো bias $w_0$।

আমরা একটি ছোট dataset ব্যবহার করব যাতে training-এর calculation সহজে দেখা যায়।

In [ ]:
import numpy as np

# Feature matrix
# Columns: [bias, engine_hp, age, mileage]
X = np.array([
    [1, 100, 20, 1200],
    [1, 150, 25,  900],
    [1, 200, 30, 1500],
    [1, 120, 40,  700],
    [1, 180, 28, 1100]
], dtype=float)

# Actual target values (car prices)
y = np.array([7.0, 8.5, 9.5, 6.3, 8.6])

print("X shape:", X.shape)
print("y shape:", y.shape)


## 3. Training-এর আসল সমস্যা

ধরি আমরা কোনো একটি $w$ বেছে নিলাম:

$$
w=
\begin{bmatrix}
w_0\\
w_1\\
w_2\\
w_3
\end{bmatrix}
$$

তাহলে prediction:

$$
\hat y=Xw
$$

এখন actual target:

$$
y
$$

এর সাথে prediction:

$$
\hat y
$$

তুলনা করব।

### Error

প্রতিটি row-এর জন্য:

$$
error=\hat y-y
$$

যেমন actual price যদি `8.0` হয় এবং prediction `7.5` হয়:

$$
error=7.5-8.0=-0.5
$$

শুধু error যোগ করলে positive এবং negative error একে অপরকে cancel করতে পারে। তাই সাধারণভাবে **squared error** ব্যবহার করি।

### Squared Error

$$
(\hat y-y)^2
$$

সব examples-এর error combine করে loss তৈরি করা হয়।

সহজভাবে:

$$
\boxed{\text{Loss} \propto \sum(\hat y-y)^2}
$$

Training-এর লক্ষ্য হলো এমন $w$ খোঁজা যাতে এই error যত ছোট সম্ভব হয়।

এই notebook-এ আমরা **Normal Equation** ব্যবহার করে সরাসরি সেই $w$ বের করব।

# 4. Normal Equation

Linear Regression-এর জন্য একটি closed-form solution আছে:

$$
\boxed{
w=(X^TX)^{-1}X^Ty
}
$$

এটাকে **Normal Equation** বলা হয়।

এখন formula-টাকে ভয় পাওয়ার দরকার নেই। আমরা একেকটা অংশ আলাদা করে বুঝব।

Formula:

$$
w=(X^TX)^{-1}X^Ty
$$

এখানে চারটি মূল operation:

1. $X^T$ বের করা
2. $X^TX$ করা
3. $(X^TX)^{-1}$ বের করা
4. $X^Ty$ করা
5. সবশেষে দুইটা result multiply করে $w$ পাওয়া

# 5. প্রথম ধাপ — Transpose

আমাদের $X$:

$$
X\text{ shape}=5\times4
$$

অর্থাৎ:

- 5 rows = 5 cars
- 4 columns = bias + 3 features

Transpose করলে rows এবং columns exchange হয়:

$$
X^T\text{ shape}=4\times5
$$

Python:

```python
Xt = X.T
```

### কেন transpose দরকার?

আমরা চাই:

$$
X^TX
$$

এখানে dimensions:

$$
(4\times5)(5\times4)
\rightarrow
4\times4
$$

মাঝের `5` match করে, তাই multiplication করা যায়।

Transpose-এর একটি গুরুত্বপূর্ণ কাজ হলো $X$-কে এমন orientation-এ আনা যাতে feature-wise information নিয়ে square matrix তৈরি করা যায়।

In [ ]:
Xt = X.T

print("X shape :", X.shape)
print("Xt shape:", Xt.shape)
print("\nX =")
print(X)

print("\nX.T =")
print(Xt)


# 6. দ্বিতীয় ধাপ — $X^TX$

এখন:

$$
X^TX
$$

Python:

```python
XTX = X.T.dot(X)
```

Dimensions:

$$
(4\times5)(5\times4)
=
4\times4
$$

অর্থাৎ:

```text
X   → 5 × 4
X.T → 4 × 5
        ↓
X.T @ X → 4 × 4
```

এই matrix-টি পরে invert করব।

In [ ]:
XTX = X.T.dot(X)

print("XTX shape:", XTX.shape)
print("\nXTX =")
print(XTX)


# 7. তৃতীয় ধাপ — Inverse

আমাদের formula-তে আছে:

$$
(X^TX)^{-1}
$$

এখানে `-1` মানে ordinary power না। এটি **matrix inverse**।

Python-এ:

```python
XTX_inv = np.linalg.inv(XTX)
```

যদি:

$$
A^{-1}
$$

হয়, তাহলে:

$$
AA^{-1}=I
$$

যেখানে $I$ হলো identity matrix।

আমাদের ক্ষেত্রে:

$$
XTX\cdot(XTX)^{-1}=I
$$

### গুরুত্বপূর্ণ

Matrix inverse সব matrix-এর জন্য exist করে না। $X^TX$ যদি **singular** হয়, তাহলে `np.linalg.inv()` error দিতে পারে।

আর বাস্তব ML code-এ সরাসরি inverse না করে অনেক সময় `np.linalg.solve()` ব্যবহার করা বেশি numerically stable। কিন্তু Normal Equation বোঝার জন্য inverse form-টাই আগে শেখা গুরুত্বপূর্ণ।

In [ ]:
XTX_inv = np.linalg.inv(XTX)

print("XTX inverse:")
print(XTX_inv)


# 8. চতুর্থ ধাপ — $X^Ty$

Formula:

$$
w=(X^TX)^{-1}X^Ty
$$

এখন:

$$
X^Ty
$$

করব।

Dimensions:

$$
X^T=(4\times5)
$$

এবং:

$$
y=(5\times1)
$$

তাই:

$$
(4\times5)(5\times1)
=
4\times1
$$

Python:

```python
XTy = X.T.dot(y)
```

অর্থাৎ:

```text
X.T → 4 × 5
y   → 5 × 1
       ↓
X.T @ y → 4 × 1
```

In [ ]:
XTy = X.T.dot(y)

print("XTy shape:", XTy.shape)
print("\nXTy =")
print(XTy)


# 9. সবকিছু একসাথে — $w$ বের করি

এখন formula:

$$
\boxed{
w=(X^TX)^{-1}X^Ty
}
$$

আমাদের কাছে already আছে:

```python
XTX_inv
XTy
```

তাই:

```python
w = XTX_inv.dot(XTy)
```

এটাই **trained weights**।

অর্থাৎ training-এর পরে model-এর কাছে এমন weights থাকবে যেগুলো $Xw$-কে actual $y$-এর কাছাকাছি নিয়ে যায়।

In [ ]:
w = XTX_inv.dot(XTy)

print("Trained weights:")
print(w)


# 10. Training-এর পরে Prediction

এখন আমাদের trained `w` আছে।

আগের lesson-এর formula মনে করো:

$$
\hat y=Xw
$$

তাই:

```python
predictions = X.dot(w)
```

এখন model-এর predictions বের হবে।

In [ ]:
predictions = X.dot(w)

print("Actual values:")
print(y)

print("\nPredictions:")
print(predictions)


# 11. Actual বনাম Prediction

ধারণাটা খুব simple:

```text
Actual y
   ↓
[real prices]

      compare

Prediction ŷ = Xw
   ↓
[model predicted prices]
```

Training এমন `w` খুঁজছে যাতে:

$$
\boxed{Xw\approx y}
$$

অর্থাৎ prediction এবং actual value-এর difference কম হয়।

এই কারণেই training-এর মূল কাজ হলো **weights শেখা**।

In [ ]:
errors = predictions - y
squared_errors = errors ** 2
mse = squared_errors.mean()

print("Errors:")
print(errors)

print("\nSquared errors:")
print(squared_errors)

print("\nMSE:", mse)


# 12. পুরো Training Code — একসাথে

এখন সব step একসাথে লিখি।

```python
X = feature matrix
y = actual target

w = (X.T @ X)^(-1) @ X.T @ y
```

তারপর:

```python
predictions = X @ w
```

অর্থাৎ পুরো flow:

$$
X,y
\longrightarrow
w
\longrightarrow
Xw
\longrightarrow
\hat y
$$

### Code

In [ ]:
import numpy as np

def train_linear_regression(X, y):
    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    XTy = X.T.dot(y)

    w = XTX_inv.dot(XTy)

    return w


def predict(X, w):
    return X.dot(w)


# Train
w = train_linear_regression(X, y)

# Predict
predictions = predict(X, w)

print("Weights:")
print(w)

print("\nPredictions:")
print(predictions)

print("\nActual:")
print(y)


# 13. সবচেয়ে গুরুত্বপূর্ণ Dimension Check

এগুলো মুখস্থ না করে shape মিলিয়ে দেখো:

$$
X:(m\times(n+1))
$$

এখানে:
- $m$ = number of examples
- $n+1$ = bias + features

তাহলে:

$$
X^T:(n+1)\times m
$$

এখন:

$$
X^TX:
((n+1)\times m)(m\times(n+1))
$$

তাই:

$$
\boxed{X^TX:(n+1)\times(n+1)}
$$

এবং:

$$
X^Ty:
((n+1)\times m)(m\times1)
$$

তাই:

$$
\boxed{X^Ty:(n+1)\times1}
$$

শেষে:

$$
((n+1)\times(n+1))
((n+1)\times1)
=
(n+1)\times1
$$

অর্থাৎ output হলো weight vector $w$।

# 14. একটা খুব Important Connection

আগের lesson:

$$
\boxed{\hat y=Xw}
$$

পরের lesson-এর training:

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

দুটো একসাথে করলে:

```text
Training:
X + y
  ↓
Normal Equation
  ↓
w

Prediction:
X + w
  ↓
Xw
  ↓
ŷ
```

অর্থাৎ:

> **Training শেখায় `w` কী হবে। Prediction সেই `w` ব্যবহার করে `ŷ` বের করে।**

এটাই এই অংশের সবচেয়ে গুরুত্বপূর্ণ concept।

# 15. Final Cheat Sheet

### Prediction

$$
\boxed{\hat y=Xw}
$$

### Training — Normal Equation

$$
\boxed{w=(X^TX)^{-1}X^Ty}
$$

### NumPy

```python
Xt = X.T
XTX = Xt.dot(X)
XTX_inv = np.linalg.inv(XTX)
XTy = Xt.dot(y)

w = XTX_inv.dot(XTy)
```

তারপর:

```python
predictions = X.dot(w)
```

### Mental Model

```text
Actual data
X + y
   ↓
"Which weights make predictions close to y?"
   ↓
Normal Equation
   ↓
trained weights w
   ↓
X.dot(w)
   ↓
predictions ŷ
```

**Training = weights খোঁজা**

**Prediction = weights ব্যবহার করে output বের করা**